# Project 3. Do better-than-expected earnings come with higher stock returns? My industry and a comparison industry
**Hogwarts Consulting** | Analyst: Vandana Kumari | Industry: SIC 37 Transportation Equipment (client: Ford Motor Company) | Comparison industry: SIC 60 Depository Institutions

**Question.** When a company announces better-than-expected earnings (high SUE), does its stock earn a higher abnormal return around the announcement (CAR)? Is this the same in my industry and in the comparison industry?

| Measure | Formula |
|---|---|
| SUE | (EPS this quarter - EPS four quarters ago) / stock price one quarter ago |
| Abnormal return (AR), one day | stock return - market return |
| CAR | sum of the AR on day -1, day 0 and day +1. Day 0 = the first trading day on or after the announcement date (`rdq`) |

**Rules (the same for both industries).** Announcement date known and between Jan 2021 and Dec 2024 | SUE can be computed | last quarter's price at least $5 | |SUE| at most 1 | all 3 daily abnormal returns found.

**How this notebook works.** Steps 1 to 11 are my Project 2 steps. The only change is that the link file keeps **both industries** (`SICS`), so the same code and the same rules run on both. Every table keeps a column `sic2`, so I can count the rows of each industry after every merge and rule. The three SUE groups are made **inside each industry**.

| Part | What |
|---|---|
| Steps 1 to 11 | Build the final table (one row = one company-quarter announcement) for both industries |
| Part 3 | Checks: row counts, duplicates, dates, extremes, same result as Project 2 for SIC 37, one hand check |
| Part 4 | Figure 1 (groups), t-test, regressions, Figure 2 (scatter), reasons for Figure 3 |

In [ ]:
# Session verification (class homework idea).
import subprocess
from datetime import datetime

print("=" * 40)
print("SESSION VERIFICATION")
print("=" * 40)
print("Run time:", datetime.now())
print("Colab session ID:", subprocess.getoutput("hostname"))
print("=" * 40)

student_name = input("Enter your full name: ")
print("\nSubmitted by:", student_name)

SESSION VERIFICATION
Run time: 2026-11-23 16:19:15.737393
Colab session ID: wrds-hnt-05-w.wharton.private


In [ ]:
# Setup. Libraries and settings (source: class homework notebook; I changed the settings to quarterly and daily)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import wrds

conn = wrds.Connection()    # log in to WRDS

# Settings. If I want to change something, I change it here only.
SICS          = ["37", "60"]     # my industry (Transportation Equipment) and my comparison industry (Depository Institutions)
CLIENT_TICKER = "F"              # my client: Ford Motor Company
CLIENT_NAME   = "FORD MOTOR"     # part of the company name in the link file
COMP_TICKER   = "JPM"            # a well-known company of the comparison industry, only for one hand check
COMP_NAME     = "JPMORGAN"       # part of its name in the link file (if it is not found, the hand check uses another company)
PULL_START    = "2019-06-01"     # quarters ending after this date; the earlier quarters give EPS four quarters ago
PULL_END      = "2024-12-31"     # last quarter-end date pulled
ANN_START     = "2021-01-01"     # announcements from this date ...
ANN_END       = "2024-12-31"     # ... to this date
RET_START     = "2020-12-01"     # daily returns: a little before the first and after the last announcement
RET_END       = "2025-01-31"
MIN_PRC       = 5                # rule: drop a quarter if last quarter's price is below this
MAX_SUE       = 1                # rule: drop a quarter if |SUE| is above this

# Row-count log. Every merge and every rule adds one line here. It is the evidence for Check 1.
funnel = []
def log(step, what, before, after):
    # before and after are tables with a column sic2, so I can count the rows of each industry
    row = {"Step": step, "What": what}
    text = step + ": " + what
    for sic in SICS:
        n_b = int((before["sic2"] == sic).sum())
        n_a = int((after["sic2"] == sic).sum())
        row["SIC " + sic + " before"] = n_b
        row["SIC " + sic + " after"] = n_a
        text = text + " | SIC " + sic + ": " + str(n_b) + " -> " + str(n_a)
    funnel.append(row)
    print(text)

# Colors and names for the figures: the same color for each industry everywhere.
COLORS = {"37": "#2E86AB", "60": "#E0A458"}      # the same colors as the class notebook (w12 in-class)
NAMES  = {"37": "Transportation Equipment (SIC 37)", "60": "Depository Institutions (SIC 60)"}
LABELS = ["Low", "Middle", "High"]       # the three SUE groups

## Step 1. Link file (ccm_link.csv), my two industries and my client
The link file connects the Compustat company id (`gvkey`) to the CRSP id (`permno`) and says between which dates the link is valid. One gvkey can have several rows, so dates matter when I merge.

In [ ]:
# Step 1. Read the link file, keep my two industries, find my client (source: class homework Step 1; the client lookup is my addition)

# 1a. Read every column as text so nothing is changed or guessed.
link_all = pd.read_csv("ccm_link.csv", dtype=str)

# 1b. gvkey has 6 characters (for example 004988); put back leading zeros that Excel may have dropped.
link_all["gvkey"] = link_all["gvkey"].str.zfill(6)

# 1c. permno is a number.
link_all["permno"] = pd.to_numeric(link_all["permno"]).astype(int)

# 1d. The date columns become real dates. errors="coerce" turns a bad value into a missing date.
link_all["linkdt"]    = pd.to_datetime(link_all["linkdt"], format="mixed", errors="coerce")
link_all["linkenddt"] = pd.to_datetime(link_all["linkenddt"], format="mixed", errors="coerce")

# 1e. A link with no end date is still valid, so it gets a far-future end date.
link_all["linkenddt"] = link_all["linkenddt"].fillna(pd.Timestamp("2099-12-31"))

# 1f. Keep only the companies of my two industries.
link = link_all[link_all["sic2"].isin(SICS)].reset_index(drop=True)
for sic in SICS:
    if sic not in set(link["sic2"]):
        raise ValueError("SIC " + sic + " is not in ccm_link.csv. Industries in the file: " + str(sorted(link_all["sic2"].unique())))
    one = link[link["sic2"] == sic]
    print("SIC", sic, "| link rows:", len(one), "| companies (gvkey):", one["gvkey"].nunique())

# 1h. Which industry each company and each stock belongs to (used to count rows by industry).
sic_of_gvkey  = link.drop_duplicates("gvkey").set_index("gvkey")["sic2"]
sic_of_permno = link.drop_duplicates("permno").set_index("permno")["sic2"]
print("companies that appear in both industries:", int(link.groupby("gvkey")["sic2"].nunique().gt(1).sum()), "(should be 0)")

# 1g. Find my client in the link file by ticker and company name.
is_client = (link["ticker"] == CLIENT_TICKER) & link["comnam"].str.contains(CLIENT_NAME, case=False, na=False)
client_link = link[is_client]
client_gvkeys = list(client_link["gvkey"].unique())
print("client gvkey(s):", client_gvkeys)
if len(client_gvkeys) == 0:
    print("WARNING: my client was not found in the link file. Check CLIENT_TICKER and CLIENT_NAME in the Setup cell.")
client_link

## Step 2. Quarterly earnings from Compustat (comp.fundq)
One row of `comp.fundq` = one company and one fiscal quarter. The four `indfmt / datafmt / popsrc / consol` lines are the standard Compustat filters that keep one standard row per company and quarter. `rdq` is the day the company announced the quarter's earnings (this is day 0); it can be missing.

In [ ]:
# Step 2. Pull the quarterly data (source: class homework Step 2; I changed funda to fundq and added fqtr and rdq)

# 2a. The SQL query needs the gvkeys as one text:  '001690','002176',...
gv = "'" + "','".join(link["gvkey"].unique()) + "'"

# 2b. Ask WRDS for only the rows and columns I need.
fq_raw = conn.raw_sql(f"""
    select gvkey, datadate, fyearq, fqtr, rdq, epspxq, prccq
    from comp.fundq
    where gvkey in ({gv})
      and datadate between '{PULL_START}' and '{PULL_END}'
      and indfmt = 'INDL' and datafmt = 'STD' and popsrc = 'D' and consol = 'C'
""", date_cols=["datadate", "rdq"])

# 2c. Add the industry of every company (from the link file), so I can count the rows of each industry.
fq_raw["gvkey"] = fq_raw["gvkey"].astype(str).str.zfill(6)
fq_raw["sic2"] = fq_raw["gvkey"].map(sic_of_gvkey)

print("company-quarters pulled from Compustat:", len(fq_raw))
print(fq_raw["sic2"].value_counts().to_string())
fq_raw.head()

## Step 3. Data types, missing values, quarter number
WRDS can return "nullable" types (Float64, Int64). I turn them into plain numbers (a missing value becomes NaN) and count what is missing. Then I give every quarter a **quarter number = fiscal year x 4 + fiscal quarter**, so that "four quarters ago" is simply the number minus 4 (the idea from the in-class notebook).

In [ ]:
# Step 3. Fix types, count missing values, add the quarter number (source: class homework Step 2c and in-class quarter-number notebook)
print(fq_raw.dtypes)

fq = fq_raw.copy()
fq["gvkey"]   = fq["gvkey"].astype(str).str.zfill(6)
fq["epspxq"]  = pd.to_numeric(fq["epspxq"], errors="coerce").astype("float64")
fq["prccq"]   = pd.to_numeric(fq["prccq"], errors="coerce").astype("float64")
fq["fyearq"]  = pd.to_numeric(fq["fyearq"], errors="coerce").astype("float64")
fq["fqtr"]    = pd.to_numeric(fq["fqtr"], errors="coerce").astype("float64")

# How much is missing in each column?
missing = pd.DataFrame({"missing": fq.isna().sum(), "missing %": (fq.isna().mean() * 100).round(1)})
print("\nmissing values:")
print(missing)

# A quarter without a fiscal year or quarter number cannot be placed in time, so it is dropped (and counted).
n_nonum = fq[["fyearq", "fqtr"]].isna().any(axis=1).sum()
print("\nquarters without a fiscal year or quarter number (dropped):", n_nonum)
fq = fq.dropna(subset=["fyearq", "fqtr"])
fq["fyearq"] = fq["fyearq"].astype(int)
fq["fqtr"]   = fq["fqtr"].astype(int)

# Quarter number: 2021 Q1 = 2021*4 + 1 = 8085, 2021 Q2 = 8086, ... 2022 Q1 = 8089
fq["qnum"] = fq["fyearq"] * 4 + fq["fqtr"]

# One row per company-quarter: keep one row if a (gvkey, datadate) or (gvkey, quarter number) appears twice.
fq = fq.sort_values(["gvkey", "datadate"])
n_before = len(fq)
fq = fq.drop_duplicates(["gvkey", "datadate"], keep="last")
fq = fq.drop_duplicates(["gvkey", "qnum"], keep="last").reset_index(drop=True)
print("duplicate company-quarters removed:", n_before - len(fq))
print("company-quarters after cleaning:", len(fq))
fq.head()

## Step 4. Merge 1: add the CRSP id (permno)
Merge `fundq` with the link file on `gvkey` (inner join). A gvkey with two link rows now appears twice, so I keep a link row only if the **quarter-end date (`datadate`) is inside the link dates**. Then each company-quarter should be on one row.

In [ ]:
# Step 4. Merge 1: fundq + link file on gvkey, then the link date rule (source: class homework Step 3a and 3b)
panel = fq.merge(link[["gvkey", "permno", "ticker", "comnam", "linkdt", "linkenddt"]], on="gvkey", how="inner")
print("after the merge on gvkey (a gvkey with 2 links is now on 2 rows):", len(panel))

# Keep the link only if it was valid on the quarter-end date. True/False for each row.
valid = (panel["linkdt"] <= panel["datadate"]) & (panel["datadate"] <= panel["linkenddt"])
panel = panel[valid]

# If one company-quarter still appears twice (two valid links), keep one row.
print("rows that repeat a gvkey + datadate before I drop them:", panel.duplicated(["gvkey", "datadate"]).sum())
panel = panel.drop_duplicates(["gvkey", "datadate"], keep="last")
panel = panel.drop(columns=["linkdt", "linkenddt"]).reset_index(drop=True)

log("Merge 1", "comp.fundq + ccm_link.csv (inner, gvkey, link dates)", fq, panel)
print("companies:", panel["gvkey"].nunique(), "| client rows:", panel["gvkey"].isin(client_gvkeys).sum())

## Step 5. Merge 2 and Merge 3: EPS four quarters ago, price one quarter ago, SUE
A row of `fundq` only has its own quarter. To get an older value I make a **copy of the table in which the quarter number is moved forward** (+4 for EPS, +1 for price) and merge it back on `gvkey + quarter number`. The merge is `left`, so a quarter with no history stays in the table (its value is NaN) and I can count it. Then I check that "four quarters ago" really is about a year earlier (a skipped quarter would otherwise pull a wrong row).

In [ ]:
# Step 5. Merge 2 and Merge 3: add last year's EPS and last quarter's price, compute SUE (source: in-class quarter-number notebook Part A; I added the date-gap check from the homework)

# --- Merge 2: EPS four quarters ago. The copy comes from Compustat (fq), moved forward by 4 quarters.
eps4 = fq[["gvkey", "qnum", "datadate", "epspxq"]].copy()
eps4["qnum"] = eps4["qnum"] + 4
eps4 = eps4.rename(columns={"datadate": "date_4q_ago", "epspxq": "eps_4q_ago"})

before = panel
panel = panel.merge(eps4, on=["gvkey", "qnum"], how="left", validate="one_to_one")
log("Merge 2", "+ EPS four quarters ago (left, gvkey + quarter number)", before, panel)

# --- Merge 3: price one quarter ago, moved forward by 1 quarter.
prc1 = fq[["gvkey", "qnum", "datadate", "prccq"]].copy()
prc1["qnum"] = prc1["qnum"] + 1
prc1 = prc1.rename(columns={"datadate": "date_1q_ago", "prccq": "prc_1q_ago"})

before = panel
panel = panel.merge(prc1, on=["gvkey", "qnum"], how="left", validate="one_to_one")
log("Merge 3", "+ price one quarter ago (left, gvkey + quarter number)", before, panel)

# --- Check that the old row really is about 4 quarters (330-400 days) and 1 quarter (60-125 days) earlier.
gap4 = (panel["datadate"] - panel["date_4q_ago"]).dt.days
gap1 = (panel["datadate"] - panel["date_1q_ago"]).dt.days
bad4 = panel["eps_4q_ago"].notna() & ~gap4.between(330, 400)
bad1 = panel["prc_1q_ago"].notna() & ~gap1.between(60, 125)
print("EPS four quarters ago set to missing (date gap not about a year):", bad4.sum())
print("price one quarter ago set to missing (date gap not about a quarter):", bad1.sum())
panel.loc[bad4, "eps_4q_ago"] = np.nan
panel.loc[bad1, "prc_1q_ago"] = np.nan

# --- SUE = (EPS this quarter - EPS four quarters ago) / price one quarter ago
panel["sue"] = (panel["epspxq"] - panel["eps_4q_ago"]) / panel["prc_1q_ago"]

print("\ncompany-quarters:", len(panel), "| SUE can be computed:", panel["sue"].notna().sum())
panel[["ticker", "datadate", "rdq", "epspxq", "eps_4q_ago", "prc_1q_ago", "sue"]].head(8)

## Step 6. Rules 1 to 5 (company-quarter table, before any daily return is pulled)
Each rule is a True/False test on every row. I apply them **one at a time** and count the rows after each. A missing value never passes a rule (`NaN >= 5` is False), so that row is dropped. I do not fill missing values with 0. I also print how many rows of my client are left after each rule, so I can see where my client would drop out.

In [ ]:
# Step 6. Rules 1-5, one at a time, with the row count after each (source: class homework Step 6 and in-class funnel; the client count is my addition)
def apply_rule(df, step, what, passes):
    out = df[passes]
    log(step, what, df, out)
    print("          client rows left:", out["gvkey"].isin(client_gvkeys).sum())
    return out

events = panel.copy()
print("start: client rows =", events["gvkey"].isin(client_gvkeys).sum())

events = apply_rule(events, "Rule 1", "announcement date (rdq) is known",
                    events["rdq"].notna())
events = apply_rule(events, "Rule 2", "announcement date Jan 2021 - Dec 2024",
                    events["rdq"].between(ANN_START, ANN_END))
events = apply_rule(events, "Rule 3", "SUE can be computed",
                    events["sue"].notna())
events = apply_rule(events, "Rule 4", "last quarter's price >= $" + str(MIN_PRC),
                    events["prc_1q_ago"] >= MIN_PRC)
events = apply_rule(events, "Rule 5", "|SUE| <= " + str(MAX_SUE),
                    events["sue"].abs() <= MAX_SUE)

events = events.reset_index(drop=True)
print("\ncompany-quarters in the table:", len(events), "| companies:", events["gvkey"].nunique())

## Step 7. Daily stock and market returns (crsp.dsf, crsp.dsi), trading-day numbers
`dsf` = one stock on one trading day, `dsi` = the market on one trading day. I pull only my stocks and dates (`dsf` is very big). The market is not open every calendar day, so I number only the days when the market was open: 1, 2, 3, ... Then "the day before" and "the day after" are simply the number -1 and +1 (the idea from the in-class notebook Part B).

In [ ]:
# Step 7. Pull daily returns and number the trading days (source: class homework Step 4 and in-class day-0 notebook; I changed monthly to daily)

# 7a. The stocks I need: the permno of every company-quarter that passed rules 1-5.
permnos = ",".join(str(int(p)) for p in events["permno"].unique())

# 7b. Daily stock returns (dsf) and daily market returns (dsi).
stock = conn.raw_sql(f"""select permno, date, ret, prc from crsp.dsf
                         where permno in ({permnos}) and date between '{RET_START}' and '{RET_END}'""",
                     date_cols=["date"])
mkt = conn.raw_sql(f"""select date, vwretd from crsp.dsi
                       where date between '{RET_START}' and '{RET_END}'""",
                   date_cols=["date"])

# 7c. The industry of every stock, then plain numbers (a missing return becomes NaN). prc is negative when CRSP shows a bid/ask average, so I use abs().
stock["permno"] = stock["permno"].astype(int)
stock["sic2"]   = stock["permno"].map(sic_of_permno)
stock["ret"]    = pd.to_numeric(stock["ret"], errors="coerce").astype("float64")
stock["prc"]    = pd.to_numeric(stock["prc"], errors="coerce").astype("float64").abs()
mkt["vwretd"]   = pd.to_numeric(mkt["vwretd"], errors="coerce").astype("float64")

# 7d. Trading-day numbers: the market calendar, one row per trading day, numbered 1, 2, 3, ...
cal = mkt.sort_values("date").drop_duplicates("date").reset_index(drop=True)
cal["tday"] = cal.index + 1

print("stock-days pulled:", len(stock), "| stocks:", stock["permno"].nunique())
print("trading days in the market table:", len(cal), "| from", cal["date"].min().date(), "to", cal["date"].max().date())
cal.head()

## Step 8. Merge 4: find day 0
Day 0 = the **first trading day on or after the announcement date**. I pair every announcement with every trading day, keep the days on or after `rdq`, and take the smallest trading-day number. If the announcement is on a Saturday, day 0 is the next Monday.

In [ ]:
# Step 8. Merge 4: day 0 for every announcement (source: in-class day-0 notebook Part B, Step 1)
ann = events.copy()
ann["ann_id"] = ann.index                  # one number for each company-quarter announcement

# Pair every announcement with every trading day, keep the days on or after rdq ...
pairs = ann[["ann_id", "rdq"]].merge(cal[["date", "tday"]], how="cross")
pairs = pairs[pairs["date"] >= pairs["rdq"]]

# ... and take the first one: the smallest trading-day number (and its date) for each announcement.
day0 = pairs.groupby("ann_id").agg(day0=("tday", "min"), day0_date=("date", "min")).reset_index()

before = ann
ann = ann.merge(day0, on="ann_id", how="left", validate="one_to_one")
ann["day0"] = ann["day0"].astype("Int64")      # whole numbers, NaN-safe
log("Merge 4", "announcements + trading days (day 0 = first trading day on or after rdq)", before, ann)
print("announcements without a day 0 (rdq after the last trading day):", ann["day0"].isna().sum())
ann[["ticker", "datadate", "rdq", "day0", "day0_date"]].head()

## Step 9. Merge 5: abnormal return of every stock-day
Add the market return of the same date to every stock-day (`left`: every stock-day stays, and a missing market return would show as NaN). Abnormal return = stock return - market return.

In [ ]:
# Step 9. Merge 5: dsf + dsi on date, then AR = stock return - market return (source: class homework Step 4e/4f; I changed month number to date)
before = stock
stock = stock.merge(cal[["date", "tday", "vwretd"]], on="date", how="left", validate="many_to_one")
log("Merge 5", "dsf + dsi (left, date): market return added to every stock-day", before, stock)

stock["ar"] = stock["ret"] - stock["vwretd"]
print("stock-days with a missing return (ret):", int(stock["ret"].isna().sum()))
print("stock-days with a missing market return:", int(stock["vwretd"].isna().sum()))
print("stock-days that are not a market trading day:", int(stock["tday"].isna().sum()))

# Keep only stock-days that have a trading-day number (the others cannot be matched).
stock = stock.dropna(subset=["tday"]).copy()
stock["tday"] = stock["tday"].astype("Int64")
stock[["permno", "date", "tday", "ret", "vwretd", "ar", "prc"]].head()

## Step 10. Merge 6: the 3-day window, CAR and the number of days found
Every announcement gets three rows: day -1, day 0, day +1 (trading-day number = day 0 + offset). I look up the abnormal return of each of these days (`left`: the row stays even if the day is missing). CAR = the sum of the 3 abnormal returns. `sum()` skips missing values quietly, so I **count** the days (`days_found`), exactly like `months_found` in the homework.

In [ ]:
# Step 10. Merge 6: window of 3 trading days, look up AR, add up to CAR, count the days (source: in-class day-0 notebook Part B, Steps 2-4)
offsets = pd.DataFrame({"offset": [-1, 0, 1]})
window = ann[["ann_id", "permno", "sic2", "rdq", "day0"]].merge(offsets, how="cross")
window["tday"] = window["day0"] + window["offset"]

# Look up the stock-day with the same permno and trading-day number. left = keep the row even if the day is missing.
before = window
window = window.merge(stock[["permno", "tday", "date", "ret", "vwretd", "ar", "prc"]],
                      on=["permno", "tday"], how="left", validate="many_to_one")
log("Merge 6", "window (3 days per announcement) + stock-days (left, permno + trading day)", before, window)

# CAR = sum of the 3 abnormal returns; days_found = how many of the 3 have a return.
car = window.groupby("ann_id")["ar"].agg(["sum", "count"]).reset_index()
car = car.rename(columns={"sum": "car", "count": "days_found"})
ann = ann.merge(car, on="ann_id", how="left", validate="one_to_one")

print("how many of the 3 days have a return, per announcement:")
print(ann["days_found"].value_counts().sort_index())
window[["ann_id", "offset", "tday", "date", "ret", "vwretd", "ar"]].head(6)

## Step 11. Rule 6 and the final table (both industries)
Keep only announcements where all 3 days have a return. **One row of the final table = one company-quarter announcement. Primary key = `gvkey` + `datadate`.**

In [ ]:
# Step 11. Rule 6 and the final table (source: class homework Step 6a, the months_found rule, changed to days_found == 3)
before = ann
final = ann[ann["days_found"] == 3]
log("Rule 6", "all 3 daily abnormal returns found (days -1, 0, +1)", before, final)

# Sorted, so that the groups in the results are always the same when I run the notebook again.
final = final.sort_values(["gvkey", "rdq"]).reset_index(drop=True)
print("final table: one row = one company-quarter announcement")
print("client announcements:", final["gvkey"].isin(client_gvkeys).sum())
final.groupby("sic2")[["sue", "car"]].describe().T

# Part 3. Checks
I check the new industry (SIC 60) the same way as in Project 2, and I check that the code still gives my Project 2 result for SIC 37.

## Check 1. Row counts before and after every merge and rule (both industries)

In [ ]:
# Check 1. Rows before and after every merge and every rule, for each industry, in the order of my code
check1 = pd.DataFrame(funnel)
for sic in SICS:
    check1["SIC " + sic + " change"] = check1["SIC " + sic + " after"] - check1["SIC " + sic + " before"]
print(check1.to_string(index=False))

In [ ]:
# Check 1 (continued). Which announcements did Rule 6 remove, and why? days_found = how many of the 3 daily abnormal returns were found.
lost = ann[ann["days_found"] < 3]                                  # the announcements that Rule 6 removed
print("announcements removed by Rule 6, by industry:")
print(lost.groupby("sic2").size().to_string())
print("\ndays_found of those announcements (0 = no daily return at all around day 0):")
print(lost.groupby(["sic2", "days_found"]).size().to_string())

# For each stock: how many announcements were lost, and how many stock-days and which dates the CRSP pull has for it
info = lost.groupby(["sic2", "ticker", "permno"]).size().rename("announcements lost").reset_index()
info["stock-days in my CRSP pull"] = info["permno"].map(stock.groupby("permno").size()).fillna(0).astype(int)
info["first date"] = info["permno"].map(stock.groupby("permno")["date"].min())
info["last date"] = info["permno"].map(stock.groupby("permno")["date"].max())
print()
print(info.to_string(index=False))

## Check 2. Duplicates
The key of the final table is `gvkey` + `datadate`. It must appear only once.

In [ ]:
# Check 2. Does any key appear more than once? duplicated() marks a row True if an earlier row has the same key; sum() counts the True values.
print("duplicate keys (gvkey + datadate):", final.duplicated(["gvkey", "datadate"]).sum(), "(it must be 0)")
print("duplicate keys (gvkey + quarter number):", final.duplicated(["gvkey", "qnum"]).sum(), "(it must be 0)")
print("duplicate (permno + announcement date):", final.duplicated(["permno", "rdq"]).sum(), "(should be 0)")
print()
for sic in SICS:
    one = final[final["sic2"] == sic]
    print("SIC", sic, "| rows:", len(one), "| distinct keys:", len(one.drop_duplicates(["gvkey", "datadate"])), "| companies:", one["gvkey"].nunique())

## Check 3. Dates
(a) The announcement must be later than the quarter end. (b) Day 0 must be a day the market was open.

In [ ]:
# Check 3. Dates, for both industries together and then for each industry
# (a) Count the rows where the announcement date is NOT after the quarter-end date. It should be 0.
not_after = final["rdq"] <= final["datadate"]
gap = (final["rdq"] - final["datadate"]).dt.days
print("(a) rows where rdq is not after datadate:", not_after.sum(), "(should be 0)")
print("    smallest gap between quarter end and announcement (days):", gap.min(), "| largest gap (days):", gap.max())
print(gap.groupby(final["sic2"]).agg(["min", "median", "max"]).to_string())

# (b) Every day 0 must be a trading day in the market table.
trading_days = set(cal["date"])
print("\n(b) day 0 dates that are NOT in the daily market table:", (~final["day0_date"].isin(trading_days)).sum(), "(should be 0)")
print("    day 0 earlier than the announcement date:", (final["day0_date"] < final["rdq"]).sum(), "(should be 0)")

# When the announcement is not on a trading day (weekend, holiday), day 0 moves to the next trading day.
moved = final[final["day0_date"] != final["rdq"]]
print("    announcements whose day 0 is later than rdq:", len(moved))
print("    weekday of those announcements:")
print(moved["rdq"].dt.day_name().value_counts().to_string())
print("    largest gap rdq -> day 0 (calendar days):", (final["day0_date"] - final["rdq"]).dt.days.max(), "(more than about 5 would be suspicious)")

## Check 4. Extremes
The 5 largest |CAR| of each industry, and the three daily numbers behind each one: one big day (news) or a strange value (data error)?

In [ ]:
# Check 4. The 5 largest |CAR| in each industry (source: Project 2 Check 4)
cols = ["sic2", "ticker", "comnam", "datadate", "rdq", "sue", "car", "prc_1q_ago"]
top5 = {}
for sic in SICS:
    one = final[final["sic2"] == sic]
    top5[sic] = one.loc[one["car"].abs().nlargest(5).index]
    print("SIC", sic)
    print(top5[sic][cols].to_string())
    print()
print("summary of CAR by industry:")
print(final.groupby("sic2")["car"].describe().round(4).to_string())

In [ ]:
# Check 4 (continued). The daily numbers behind the 10 rows (5 for each industry)
top_ids = list(top5["37"]["ann_id"]) + list(top5["60"]["ann_id"])
names = final[final["ann_id"].isin(top_ids)][["ann_id", "ticker"]]
window[window["ann_id"].isin(top_ids)].merge(names, on="ann_id")[
    ["sic2", "ticker", "offset", "date", "prc", "ret", "vwretd", "ar"]].sort_values(["sic2", "ticker", "date"])

## Check 5. Does the code still give my Project 2 result for SIC 37?
In Project 2 the final table of SIC 37 had 985 announcements, and the average CAR of the Low, Middle and High group was -0.0216, 0.0004 and 0.0219. The code is the same, so the result must be the same.

In [ ]:
# Check 5. Compare the SIC 37 result of this notebook with my Project 2 numbers (typed from my Project 2 report)
P2_N       = 985
P2_AVG_CAR = {"Low": -0.0216, "Middle": 0.0004, "High": 0.0219}

# The three groups are made inside each industry (see Part 4); here I only need SIC 37.
one = final[final["sic2"] == "37"].copy()
one["grp"] = pd.qcut(one["sue"].rank(method="first"), 3, labels=LABELS)
now = one.groupby("grp", observed=True)["car"].mean().round(4)

print("announcements now:", len(one), "| Project 2:", P2_N, "| same:", len(one) == P2_N)
for g in LABELS:
    print(g, "| now:", now[g], "| Project 2:", P2_AVG_CAR[g], "| same:", now[g] == P2_AVG_CAR[g])

## Check 6. One hand check in the comparison industry
Same idea as Project 2: I take one announcement of a well-known company of SIC 60, show its inputs, and compare them with the company's earnings press release and with Google Finance or Yahoo Finance. If `COMP_TICKER` is not found, the code takes the SIC 60 company with the most announcements.

In [ ]:
# Check 6. One announcement of a comparison-industry company: the inputs and the CRSP prices around day 0 (my own code)
HAND_RDQ = None      # type a date like "2024-07-16" to choose the announcement; None = the company's most recent announcement

bank = final[final["sic2"] == "60"]
comp_link = link[(link["sic2"] == "60") & (link["ticker"] == COMP_TICKER) & link["comnam"].str.contains(COMP_NAME, case=False, na=False)]
comp_gvkeys = list(comp_link["gvkey"].unique())
pool = bank[bank["gvkey"].isin(comp_gvkeys)].sort_values("rdq")
if len(pool) == 0:
    other = bank["gvkey"].value_counts().index[0]
    pool = bank[bank["gvkey"] == other].sort_values("rdq")
    print("COMP_TICKER not found in the final table. Hand check with another company: gvkey", other, "| ticker", pool["ticker"].iloc[0])
print("announcements of this company in the final table:", len(pool))

hand = pool.iloc[-1] if HAND_RDQ is None else pool[pool["rdq"] == pd.Timestamp(HAND_RDQ)].iloc[0]
print("\n--- the notebook's row (compare EPS with the press release) ---")
print(hand[["ticker", "comnam", "fyearq", "fqtr", "datadate", "rdq", "epspxq", "eps_4q_ago", "prc_1q_ago", "prccq", "sue", "car"]].to_string())

# Closing prices from CRSP around day 0 (compare with Google Finance); day -2 is shown so that the day -1 return can be checked.
prices = stock[(stock["permno"] == hand["permno"]) & stock["tday"].between(hand["day0"] - 2, hand["day0"] + 1)]
print("\n--- CRSP daily numbers around day 0 (day 0 =", hand["day0_date"].date(), ") ---")
prices[["date", "prc", "ret", "vwretd", "ar"]].reset_index(drop=True)

# Part 4. Results
## The three SUE groups, inside each industry
Each industry is split into three equal groups by its own SUE (Low, Middle, High). I use `rank(method="first")` before `qcut`, as in Project 2, because many quarters have a SUE of exactly 0.

In [ ]:
# The three SUE groups, made separately for each industry (source: Project 2 results; the loop over industries is new)
final["sue_group"] = ""
for sic in SICS:
    rows = final["sic2"] == sic                                    # True for the rows of this industry
    ranks = final.loc[rows, "sue"].rank(method="first")            # 1, 2, 3, ... by SUE, ties broken by order
    final.loc[rows, "sue_group"] = pd.qcut(ranks, 3, labels=LABELS).astype(str)
final["sue_group"] = pd.Categorical(final["sue_group"], categories=LABELS, ordered=True)

# loss = 1 if EPS this quarter is below 0 (used in the regression and in the reasons for Figure 3)
final["loss"] = (final["epspxq"] < 0).astype(int)
final["fyearq"] = final["fyearq"].astype(int)       # plain whole numbers, needed for C(fyearq) in the regression

# Winsorized CAR (class method): inside each industry, CAR below its 1st percentile is set to the 1st percentile,
# CAR above its 99th percentile is set to the 99th. No row is deleted.
final["car_w"] = final["car"]
for sic in SICS:
    rows = final["sic2"] == sic
    low_cut, high_cut = final.loc[rows, "car"].quantile([0.01, 0.99])
    final.loc[rows, "car_w"] = final.loc[rows, "car"].clip(low_cut, high_cut)       # clip pulls values outside [low_cut, high_cut] to the edge
    changed = int((final.loc[rows, "car"] != final.loc[rows, "car_w"]).sum())
    print("SIC", sic, "| winsorized CAR: cut points", round(low_cut, 4), "and", round(high_cut, 4), "| rows changed:", changed, "of", int(rows.sum()))

for sic in SICS:
    one = final[final["sic2"] == sic]
    print("SIC", sic, "| announcements with a SUE of exactly 0:", int((one["sue"] == 0).sum()), "of", len(one))

results = final.groupby(["sic2", "sue_group"], observed=True).agg(
    announcements=("car", "size"),
    average_car=("car", "mean"),
    median_car=("car", "median"),
    sue_min=("sue", "min"),
    sue_max=("sue", "max"))
results["average_car_%"] = results["average_car"] * 100
results.round(4)

## Figure 1. Average CAR by SUE group, my industry and the comparison industry
Bars = average CAR in %. The line on each bar is the 95% confidence interval: average +/- 1.96 x standard error, where standard error = standard deviation / square root of n (the class method). The number of announcements is written under each group name. The title is built from the numbers.

In [ ]:
# Figure 1. Data: number, average CAR (%) and 95% confidence interval for every industry and SUE group (class method: 95% CI = 1.96 x standard error)
rows = []
for sic in SICS:
    for g in LABELS:
        car_pct = final.loc[(final["sic2"] == sic) & (final["sue_group"] == g), "car"] * 100    # CAR in %
        n = len(car_pct)
        mean = car_pct.mean()
        se = car_pct.std() / np.sqrt(n)                  # standard error of the average
        rows.append({"industry": sic, "group": g, "n": n, "mean_%": mean,
                     "ci_low_%": mean - 1.96 * se, "ci_high_%": mean + 1.96 * se})
fig1 = pd.DataFrame(rows)

# The title states the finding and is built from the numbers, so it is always right (class idea): High minus Low in each industry.
gap = {}
for sic in SICS:
    d = fig1[fig1["industry"] == sic].set_index("group")
    gap[sic] = d.loc["High", "mean_%"] - d.loc["Low", "mean_%"]
FIG1_TITLE = ("High minus Low: " + format(gap["37"], ".1f") + " pp in " + NAMES["37"] + "\n"
              + "and " + format(gap["60"], ".1f") + " pp in " + NAMES["60"])
print(FIG1_TITLE)
fig1.round(3)

In [ ]:
# Figure 1. The bar chart, saved in the folder figures/
os.makedirs("figures", exist_ok=True)
fig, ax = plt.subplots(figsize=(9, 5.5))
width = 0.38
x = np.arange(len(LABELS))
for k, sic in enumerate(SICS):
    d = fig1[fig1["industry"] == sic]
    pos = x + (k - 0.5) * width                           # left bar and right bar
    err = [d["mean_%"] - d["ci_low_%"], d["ci_high_%"] - d["mean_%"]]
    ax.bar(pos, d["mean_%"], width, yerr=err, capsize=4, color=COLORS[sic], label=NAMES[sic],
           error_kw={"elinewidth": 1.2, "ecolor": "#333333"})
    # the average written above (or under) each error bar
    for p, m, lo_, hi_ in zip(pos, d["mean_%"], d["ci_low_%"], d["ci_high_%"]):
        if m >= 0:
            ax.annotate(format(m, ".1f") + "%", (p, hi_), xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=9)
        else:
            ax.annotate(format(m, ".1f") + "%", (p, lo_), xytext=(0, -4), textcoords="offset points", ha="center", va="top", fontsize=9)
ax.axhline(0, color="black", linewidth=0.8)

# The number of announcements goes under each group name (left bar = SIC 37, right bar = SIC 60).
tick_labels = []
for g in LABELS:
    text = g
    for sic in SICS:
        n = int(fig1.loc[(fig1["industry"] == sic) & (fig1["group"] == g), "n"].iloc[0])
        text = text + "\nSIC " + sic + ": n=" + str(n)
    tick_labels.append(text)
ax.set_xticks(x)
ax.set_xticklabels(tick_labels)
ax.set_xlabel("SUE group (Low = worst earnings surprise, High = best)")
ax.set_ylabel("Average CAR (%)")
ax.set_title(FIG1_TITLE, loc="left", fontweight="bold")
ax.margins(y=0.15)
ax.grid(axis="y", color="#E3E3E3")
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.30))      # legend under the chart
fig.savefig("figures/figure1.png", dpi=200, bbox_inches="tight")
plt.show()

## Is the difference real? t-test
For each industry: is the average CAR of the High SUE group different from the Low group? The difference is High minus Low, in percentage points. `equal_var=False` means I do not assume that both groups have the same spread (Welch t-test). I read a p-value of 0.10 or less as statistically significant, as in class.

In [ ]:
# t-test: High group against Low group, for each industry
rows = []
for sic in SICS:
    high = final.loc[(final["sic2"] == sic) & (final["sue_group"] == "High"), "car"] * 100
    low  = final.loc[(final["sic2"] == sic) & (final["sue_group"] == "Low"), "car"] * 100
    test = stats.ttest_ind(high, low, equal_var=False)
    rows.append({"industry": sic, "avg High %": high.mean(), "avg Low %": low.mean(),
                 "difference (pp)": high.mean() - low.mean(),
                 "t-statistic": test.statistic, "p-value": test.pvalue,
                 "p-value (e-notation)": format(test.pvalue, ".2e"),
                 "significant at 10%": test.pvalue <= 0.10})
ttest = pd.DataFrame(rows)
ttest.round(4)

## How strong is the link? Regressions
For each industry: `CAR = a + b x SUE`. CAR and SUE are plain numbers (0.01 = 1%). **b** = how much CAR changes when SUE goes up by 1.0, so SUE +0.01 goes with CAR +0.01 x b. I run it on the original CAR and on the winsorized CAR (class method) to see whether a few extreme CARs drive the result.

In [ ]:
# Regression 1 (class Part 4b): original CAR and winsorized CAR, one industry at a time
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

rows = []
models = {}
for sic in SICS:
    d = final[final["sic2"] == sic]
    m_orig = smf.ols("car ~ sue", data=d).fit()
    m_wins = smf.ols("car_w ~ sue", data=d).fit()
    models[sic] = m_wins                         # Figure 2 uses the winsorized line
    for label, m in [("original CAR", m_orig), ("winsorized CAR", m_wins)]:
        rows.append({"industry": sic, "CAR used": label, "n": int(m.nobs), "b": m.params["sue"],
                     "p-value of b": m.pvalues["sue"], "R2": m.rsquared,
                     "CAR change if SUE +0.01 (pp)": m.params["sue"] * 0.01 * 100})
reg1 = pd.DataFrame(rows)
reg1.round(4)

**Three models side by side (class Part 4).** (1) base; (2) with year fixed effects `C(fyearq)`; (3) with year fixed effects and `loss` (loss = 1 if EPS < 0) and the interaction `sue * loss`. The standard errors are clustered by company (`gvkey`), as in class, because the same company appears in many quarters. I use the winsorized CAR, as in class. Does b stay about the same?

In [ ]:
# Regression 2 (class Part 4): three models side by side for each industry. Stars: * p<0.10, ** p<0.05, *** p<0.01.
for sic in SICS:
    d = final[final["sic2"] == sic]
    cluster = dict(cov_type="cluster", cov_kwds={"groups": d["gvkey"]})     # standard errors clustered by company
    m1 = smf.ols("car_w ~ sue", data=d).fit(**cluster)
    m2 = smf.ols("car_w ~ sue + C(fyearq)", data=d).fit(**cluster)
    m3 = smf.ols("car_w ~ sue * loss + C(fyearq)", data=d).fit(**cluster)
    print("=" * 70)
    print(NAMES[sic])
    table = summary_col([m1, m2, m3], stars=True, float_format="%.4f",
                        model_names=["(1) base", "(2) year FE", "(3) year FE + loss"],
                        regressor_order=["sue", "loss", "sue:loss"], drop_omitted=True,
                        info_dict={"R2": lambda m: format(m.rsquared, ".3f"), "n": lambda m: str(int(m.nobs))})
    print(table)

## Figure 2. SUE and CAR, one color per industry, with each industry's regression line
Each dot is one announcement. The line is the regression of CAR on SUE for that industry (winsorized CAR, class method). CAR is shown in %. The title is built from the slopes and R2.

In [ ]:
# Figure 2. Scatter plot of SUE (x) and winsorized CAR in % (y), with the regression line of each industry (class Figure 2)
# The title states the finding and is built from the slopes and R2 (class idea).
words = []
for sic in SICS:
    direction = "higher" if models[sic].params["sue"] > 0 else "lower"      # a positive slope: higher SUE goes with higher CAR
    words.append("SIC " + sic + ": higher SUE goes with " + direction + " CAR (R2 = " + format(models[sic].rsquared, ".3f") + ")")
FIG2_TITLE = words[0] + "\n" + words[1]
print(FIG2_TITLE)

fig, ax = plt.subplots(figsize=(9, 6))
for sic in SICS:
    d = final[final["sic2"] == sic]
    ax.scatter(d["sue"], d["car_w"] * 100, s=12, alpha=0.30, color=COLORS[sic], label=NAMES[sic])
    m1 = models[sic]
    xs = np.linspace(d["sue"].min(), d["sue"].max(), 50)
    ys = (m1.params["Intercept"] + m1.params["sue"] * xs) * 100       # fitted CAR in %
    ax.plot(xs, ys, color=COLORS[sic], linewidth=3)
ax.axhline(0, color="black", linewidth=0.8)
# A few extreme SUE values stretch the x-axis. I show the middle 98% of SUE (1st to 99th percentile). The lines are still the regressions of all rows.
x_low, x_high = final["sue"].quantile([0.01, 0.99])
outside = int(((final["sue"] < x_low) | (final["sue"] > x_high)).sum())
ax.set_xlim(x_low, x_high)
print("x-axis zoom:", round(x_low, 3), "to", round(x_high, 3), "| announcements outside the zoom:", outside, "of", len(final))
ax.set_xlabel("SUE (earnings surprise, as a share of the stock price); axis shows the middle 98% of SUE")
ax.set_ylabel("CAR (%, winsorized at 1% and 99%)")
ax.set_title(FIG2_TITLE, loc="left", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
fig.savefig("figures/figure2.png", dpi=200, bbox_inches="tight")
plt.show()

## Why do the industries differ? Candidate reasons
These numbers come from the data I already have. I read them, choose **one** reason, and check it in Figure 3 (the Figure 3 cell is added after I choose).

In [ ]:
# Candidate reasons: numbers by industry (and by SUE group) from the final table
cand = final.groupby("sic2").agg(
    announcements=("car", "size"),
    companies=("gvkey", "nunique"),
    share_of_loss_quarters=("loss", "mean"),
    median_price_last_quarter=("prc_1q_ago", "median"),
    median_abs_sue=("sue", lambda s: s.abs().median()))
cand["announcements_per_company"] = cand["announcements"] / cand["companies"]
print(cand.round(3).to_string())

print("\nshare of announcements in each calendar month (rows add up to 100%):")
months = final.assign(month=final["rdq"].dt.month).pivot_table(index="sic2", columns="month", values="car", aggfunc="size", fill_value=0)
print((months.div(months.sum(axis=1), axis=0) * 100).round(1).to_string())

print("\nshare of loss quarters and median price, by industry and SUE group:")
print(final.groupby(["sic2", "sue_group"], observed=True).agg(
    share_of_loss_quarters=("loss", "mean"), median_price=("prc_1q_ago", "median")).round(3).to_string())

## Figure 3. Checking my reason: loss-making companies
**My reason (to check).** In SIC 37 many of the Low-SUE quarters are quarters with a loss (EPS below 0), and a loss may make the stock fall more. In SIC 60 almost no quarter has a loss. If this reason is right, then after I take the loss quarters out, the gap between High and Low in SIC 37 should become much smaller. Panel A shows the share of loss quarters in each group. Panel B shows the average CAR of each group without the loss quarters.

In [ ]:
# Figure 3 data. For each industry and SUE group: the share of loss quarters, and the average CAR (%) of the quarters without a loss, with a 95% CI (1.96 x standard error)
rows = []
for sic in SICS:
    for g in LABELS:
        one = final[(final["sic2"] == sic) & (final["sue_group"] == g)]
        no_loss = one[one["loss"] == 0]
        car_pct = no_loss["car"] * 100
        n = len(car_pct)
        mean = car_pct.mean()
        se = car_pct.std() / np.sqrt(n)
        rows.append({"industry": sic, "group": g, "n all": len(one), "share of loss %": one["loss"].mean() * 100,
                     "n without loss": n, "mean without loss %": mean,
                     "ci_low_%": mean - 1.96 * se, "ci_high_%": mean + 1.96 * se})
fig3 = pd.DataFrame(rows)

# High minus Low without the loss quarters, and the t-test (Welch), for each industry
gap_no_loss = {}
for sic in SICS:
    high = final.loc[(final["sic2"] == sic) & (final["sue_group"] == "High") & (final["loss"] == 0), "car"] * 100
    low  = final.loc[(final["sic2"] == sic) & (final["sue_group"] == "Low") & (final["loss"] == 0), "car"] * 100
    test = stats.ttest_ind(high, low, equal_var=False)
    gap_no_loss[sic] = high.mean() - low.mean()
    print("SIC", sic, "| High minus Low without loss quarters:", round(gap_no_loss[sic], 2), "pp | with all quarters (Figure 1):", round(gap[sic], 2), "pp | t =", round(test.statistic, 2), "| p =", format(test.pvalue, ".2e"))

# The title states the finding and is built from the numbers
FIG3_TITLE = ("Without loss quarters, High minus Low is " + format(gap_no_loss["37"], ".1f") + " pp in SIC 37 (before: " + format(gap["37"], ".1f") + ")\n"
              + "and " + format(gap_no_loss["60"], ".1f") + " pp in SIC 60 (before: " + format(gap["60"], ".1f") + ")")
print(FIG3_TITLE)
fig3.round(3)

In [ ]:
# Figure 3. Two panels: A = share of loss quarters, B = average CAR without the loss quarters
fig, (axA, axB) = plt.subplots(1, 2, figsize=(12, 5.5))
width = 0.38
x = np.arange(len(LABELS))
for k, sic in enumerate(SICS):
    d = fig3[fig3["industry"] == sic]
    pos = x + (k - 0.5) * width
    axA.bar(pos, d["share of loss %"], width, color=COLORS[sic], label=NAMES[sic])
    for p, v in zip(pos, d["share of loss %"]):
        axA.annotate(format(v, ".0f") + "%", (p, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)
    err = [d["mean without loss %"] - d["ci_low_%"], d["ci_high_%"] - d["mean without loss %"]]
    axB.bar(pos, d["mean without loss %"], width, yerr=err, capsize=4, color=COLORS[sic], label=NAMES[sic],
            error_kw={"elinewidth": 1.2, "ecolor": "#333333"})
    for p, m, lo_, hi_ in zip(pos, d["mean without loss %"], d["ci_low_%"], d["ci_high_%"]):
        if m >= 0:
            axB.annotate(format(m, ".1f") + "%", (p, hi_), xytext=(0, 4), textcoords="offset points", ha="center", va="bottom", fontsize=9)
        else:
            axB.annotate(format(m, ".1f") + "%", (p, lo_), xytext=(0, -4), textcoords="offset points", ha="center", va="top", fontsize=9)
axB.axhline(0, color="black", linewidth=0.8)
axA.set_title("A. Share of loss quarters (EPS below 0)", loc="left", fontsize=11)
axB.set_title("B. Average CAR without the loss quarters", loc="left", fontsize=11)
axA.set_ylabel("Share of quarters with a loss (%)")
axB.set_ylabel("Average CAR (%)")
for ax in (axA, axB):
    ax.set_xticks(x)
    ax.set_xticklabels(LABELS)
    ax.set_xlabel("SUE group (Low = worst surprise, High = best)")
    ax.margins(y=0.15)
    ax.grid(axis="y", color="#E3E3E3")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axB.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(-0.1, -0.15))      # one legend under the two panels
fig.suptitle(FIG3_TITLE, x=0.01, y=1.06, ha="left", fontweight="bold")      # y above 1 keeps the title off the panel titles
fig.savefig("figures/figure3.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Save only group averages and test results (no company-level data) for the GitHub folder figures/
fig1.to_csv("figures/group_averages.csv", index=False)
fig3.to_csv("figures/figure3_numbers.csv", index=False)
ttest.to_csv("figures/ttest_results.csv", index=False)
reg1.to_csv("figures/regression_model1.csv", index=False)
print("saved:", sorted(os.listdir("figures")))

In [ ]:
conn.close()